In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)  # Make it (N,1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)




In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)




In [ ]:
# 3. Create DataLoaders

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)



In [ ]:
# 4. Print shape of one batch

sample_batch = next(iter(train_loader))
X_sample, y_sample = sample_batch
print(f"Batch X shape: {X_sample.shape}")
print(f"Batch y shape: {y_sample.shape}")


In [ ]:
# 5. Display sample images
fig, axes = plt.subplots(1, 5, figsize=(15,3))
for i in range(5):
    img = X_sample[i].permute(1,2,0).numpy()  # Convert to HWC for plotting
    axes[i].imshow(img)
    axes[i].set_title(f"Age: {y_sample[i].item():.0f}")
    axes[i].axis('off')
plt.show()


In [ ]:
# Task 1: Write your model class here:
import torch.nn as nn
import torch.optim as optim

class AgeRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(AgeRegressionModel, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.model(x)





In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)
    return running_loss / len(loader.dataset)


In [ ]:
# Task 3: Write your validation loop here:
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            running_loss += loss.item() * X_batch.size(0)
    return running_loss / len(loader.dataset)


In [ ]:
# 4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = X_train_tensor.shape[1] * X_train_tensor.shape[2] * X_train_tensor.shape[3]
model = AgeRegressionModel(input_dim).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# Task 5: Start training for 20 epochs:
num_epochs = 20
train_losses = []
val_losses = []


In [ ]:
if len(train_losses) == 0 or len(val_losses) == 0:
    raise ValueError("train_losses or val_losses is empty. Run the training loop first.")

plt.figure(figsize=(8,5))
epochs_range = range(1, len(train_losses)+1)
plt.plot(epochs_range, train_losses, label='Train Loss', marker='o')
plt.plot(epochs_range, val_losses, label='Validation Loss', marker='o')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training and Validation Loss over Epochs')
plt.legend()
plt.show()


In [ ]:
# Task 2 (Bonus): Write your code here:
model.eval()
X_sample, y_sample = next(iter(test_loader))
X_sample, y_sample = X_sample.to(device), y_sample.to(device)
with torch.no_grad():
    y_pred = model(X_sample)

ig, axes = plt.subplots(1,5, figsize=(15,3))
for i in range(5):
    img = X_sample[i].cpu().permute(1,2,0).numpy()
    axes[i].imshow(img)
    axes[i].set_title(f"Pred: {y_pred[i].item():.0f}\nActual: {y_sample[i].item():.0f}")
    axes[i].axis('off')
plt.show()
